<a href="https://colab.research.google.com/github/JulTob/Ada/blob/master/Ada_Libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ada Libraries: A Beginner's Guide with Algebraic examples.

Welcome! This notebook explores how to map mathematical concepts (specifically, an algebraic structure called a **Lattice**) into clean, professional Ada code.

Along the way, we will uncover some of the most powerful features in software engineering:
1. **Interfaces:** Separating *what* our code does from *how* it does it.
2. **Polymorphism (Generics):** Writing code once that works for many different data types.
3. **Libraries:** Packaging our code so it can be easily shared and reused by others.

This is similar to Algebraic abstractions. When you care more about the relational structure than the object themselves.

Let's get started by making sure our Ada compiler (GNAT) is installed!

In [47]:
if not installed:
    !sudo apt-get update
    !sudo apt-get install -y gnat
    installed = True

## 1. Interfaces and Generics: The Menu

In a restaurant, the **menu** tells you what you can order, but it doesn't tell you how the chef cooks it. In Ada, a `.ads` (Ada Specification) file is our menu. It's the **Interface** or **Contract**.

Furthermore, we want our Lattice to work for *any* type of data that can be compared (like numbers or true/false values). This is called **Polymorphism**. In Ada, we achieve this using **Generics**.

Below, we define a generic package. It says: "Give me any `Element`, as long as it has `<` and `>` operators, and I will give you `Meet`, `Join`, `and`, and `or` functions for it."

In [48]:
%%writefile lattice.ads

generic type Element is private;
    -- Define Element as a private type so it can be anything

    -- We need to know how to compare the elements for Meet and Join --

    with function "<" (Left, Right : Element)
        return Boolean
        is <>;
        -- Requires function "<" is in scope
        ---- <> is read as "box", and here it means "in scope"

    with function ">" (Left, Right : Element)
        return Boolean
        is <>;
        -- Requires function ">" is in scope

package Lattice is

    -- Explicit contract functions
    function Meet (A, B : Element) return Element;
    function Join (A, B : Element) return Element;

    -- Overloaded operators that will act as wrappers
    function "and" (A, B : Element) return Element;
    function "or" (A, B : Element) return Element;

    end Lattice;


Overwriting lattice.ads


## 2. The Implementation: The Kitchen

Now that we have our menu, we need the kitchen. The `.adb` (Ada Body) file is where the actual work happens.

This file is hidden from the user of your code. They don't need to know that `Meet` uses an `if A < B` statement; they just need to know it returns the minimum value. This separation keeps our code clean and easy to maintain.

In [49]:
%%writefile lattice.adb

package body Lattice is

    -- Explicit implementation of Meet (Min)
    function Meet
            (A, B : Element)
            return Element is
        begin
        if A < B then
            return A;
        else
            return B;
        end if;
        end Meet;

    -- Explicit implementation of Join (Max)
    function Join
            (A, B : Element)
            return Element is
        begin
        if A > B then
            return A;
        else
            return B;
        end if;
        end Join;

    -- Operator wrappers for elegance
    function "and"
            (A, B : Element)
            return Element is
        begin
        return Meet (A, B);
        end "and";

    function "or"
            (A, B : Element)
            return Element is
        begin
        return Join (A, B);
        end "or";

    end Lattice;


Overwriting lattice.adb


## 3. Polymorphism in Action: Testing Different Types

Because we used Generics, our Lattice package is a template. We can "instantiate" (create specific versions of) it for different types without rewriting any of the logic.

Let's create three different programs to prove our single Lattice code works for:
1. Standard **Integers**
2. **Booleans** (True/False)
3. A completely custom **Bit** type that we invent (0 or 1).

In [57]:
%%writefile test_1_integers.adb
with Ada.Text_IO; use Ada.Text_IO;
with Lattice;

procedure Test_1_Integers is

    -- Instantiate the generic Lattice package for Integer
    package Int_Lattice
        is new Lattice (Element => Integer);

    use Int_Lattice;
        -- Makes Meet, Join, "and", "or" directly visible for Integers

    X : Integer := 5;
    Y : Integer := 10;

        begin
        Put_Line
            ("--- Test 1: Integer Lattice ---");
        Put_Line
            ("Lattice Meet of 5 and 10: "
            & Integer'Image(
                Meet(X, Y)));
        Put_Line
            ("Lattice Join of 5 and 10: "
            & Integer'Image(
                Join(X, Y)));
        -- Using overloaded operators via the instantiated package
        Put_Line
            ("Lattice Meet (X and Y) of 5 and 10: "
            & Integer'Image(
                X and Y));
        Put_Line
            ("Lattice Join (X or Y) of 5 and 10: "
            & Integer'Image(
                X or Y));
        end Test_1_Integers;


Overwriting test_1_integers.adb


In [58]:
%%writefile test_2_booleans.adb
with Ada.Text_IO; use Ada.Text_IO;
with Lattice;

procedure Test_2_Booleans is

    -- Instantiate the generic Lattice package for Boolean
    package Bool_Lattice
        is new Lattice (Element => Boolean);

    use Bool_Lattice;
        -- Makes Meet, Join, "and", "or" directly visible for Booleans

    A : Boolean := True;
    B : Boolean := False;

        begin
        Put_Line
            ("--- Test 2: Boolean Lattice ---");
        Put_Line
            ("Lattice Meet of True and False: "
            & Boolean'Image(
                Meet(A, B)));
        Put_Line
            ("Lattice Join of True and False: "
            & Boolean'Image(
                Join(A, B)));
        end Test_2_Booleans;


Overwriting test_2_booleans.adb


In [59]:
%%writefile test_3_custom_bits.adb
with Ada.Text_IO; use Ada.Text_IO;
with Lattice;

procedure Test_3_Custom_Bits is

    -- 1. Define our custom binary digit type
    type Bit is range 0 .. 1;

    -- 2. Instantiate the Lattice for Bit
    -- (Since Bit is a numeric type, Ada already provides '<' and '>')
    package Bit_Lattice
        is new Lattice (Element => Bit);
    use Bit_Lattice;

    Bit_Zero : Bit := 0;
    Bit_One  : Bit := 1;

begin
    Put_Line
        ("--- Test 3: Custom Bit Lattice ---");
    Put_Line
        ("Lattice Meet (0 and 1): "
        & Bit'Image(
            Bit_Zero and Bit_One));
    Put_Line
        ("Lattice Join (0 or 1): "
        & Bit'Image(
            Bit_Zero or Bit_One));
end Test_3_Custom_Bits;


Overwriting test_3_custom_bits.adb


In [56]:
!gnatmake test_1_integers.adb
!./test_1_integers
print("\n")
!gnatmake test_2_booleans.adb
!./test_2_booleans
print("\n")
!gnatmake test_3_custom_bits.adb
!./test_3_custom_bits


x86_64-linux-gnu-gcc-10 -c test_1_integers.adb
x86_64-linux-gnu-gnatbind-10 -x test_1_integers.ali
x86_64-linux-gnu-gnatlink-10 test_1_integers.ali
--- Test 1: Integer Lattice ---
Lattice Meet (X and Y) of 5 and 10:  5
Lattice Join (X or Y) of 5 and 10:  10


x86_64-linux-gnu-gcc-10 -c test_2_booleans.adb
x86_64-linux-gnu-gnatbind-10 -x test_2_booleans.ali
x86_64-linux-gnu-gnatlink-10 test_2_booleans.ali
--- Test 2: Boolean Lattice ---
Lattice Meet of True and False: FALSE
Lattice Join of True and False: TRUE


x86_64-linux-gnu-gcc-10 -c test_3_custom_bits.adb
x86_64-linux-gnu-gnatbind-10 -x test_3_custom_bits.ali
x86_64-linux-gnu-gnatlink-10 test_3_custom_bits.ali
--- Test 3: Custom Bit Lattice ---
Lattice Meet (0 and 1):  0
Lattice Join (0 or 1):  1


## 4. Packaging for the World: Creating a Library

Compiling single files is fine for small tests, but what if you want to share your Lattice package on GitHub for others to use in massive projects? You wouldn't want them to have to recompile your `.adb` source code every single time.

Instead, we create a **Precompiled Library**. We take our Ada code and translate it into a binary file (`.a` in Linux) that contains machine code.

To do this cleanly, Ada uses **GNAT Project files (`.gpr`)**. Think of a `.gpr` file as a blueprint that tells the compiler exactly how to organize folders and build the library.

In [66]:
%%writefile lattice.gpr
library project Lattice is
    for Library_Name use "Lattice";
    for Library_Dir  use "lib";
    for Object_Dir   use "obj";
    for Source_Files use ("lattice.ads", "lattice.adb");
end Lattice;


Overwriting lattice.gpr


In [62]:
!sudo apt-get install -y gprbuild

# Create the directories required by the GPR file
!mkdir -p obj lib

# Build the library using GPRbuild
!gprbuild -P lattice.gpr

# List the contents of the lib directory to see the output
!ls -la lib/


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  gprconfig-kb libgnatprj8 libxmlada-dom7 libxmlada-input7 libxmlada-sax7
  libxmlada-schema7 libxmlada-unicode7
Suggested packages:
  gprbuild-doc
The following NEW packages will be installed:
  gprbuild gprconfig-kb libgnatprj8 libxmlada-dom7 libxmlada-input7
  libxmlada-sax7 libxmlada-schema7 libxmlada-unicode7
0 upgraded, 8 newly installed, 0 to remove and 57 not upgraded.
Need to get 3,106 kB of archives.
After this operation, 12.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libxmlada-unicode7 amd64 21.0.0-4 [49.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libxmlada-input7 amd64 21.0.0-4 [19.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libxmlada-sax7 amd64 21.0.0-4 [115 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 li

## 5. Using the Library: The Consumer

Now we have a precompiled library (`libLattice.a` inside the `lib` folder). Let's prove we can use it!

We will create a brand new Ada program that uses `Float` numbers. This program represents someone else downloading your library.

To link them together, we simply write another `.gpr` file for our new program and add the magic words: `with "lattice.gpr";`. The build tool handles all the complex wiring automatically.

In [63]:
%%writefile main_library_test.adb
with Ada.Text_IO; use Ada.Text_IO;
with Lattice;

procedure Main_Library_Test is
    -- Let's test it with a new type: Float
    package Float_Lattice is new Lattice (Element => Float);
    use Float_Lattice;

    F1 : Float := 3.14;
    F2 : Float := 2.71;
begin
    Put_Line("--- Testing Precompiled Library ---");
    Put_Line("Float Meet (Min) of 3.14 and 2.71: " & Float'Image(F1 and F2));
    Put_Line("Float Join (Max) of 3.14 and 2.71: " & Float'Image(F1 or F2));
end Main_Library_Test;


Writing main_library_test.adb


In [67]:
%%writefile test_library.gpr
-- Import our compiled Lattice library project
with "lattice.gpr";

project Test_Library is
    for Source_Files use ("main_library_test.adb");
    for Main use ("main_library_test.adb");
    for Object_Dir use "obj_test";
end Test_Library;


Overwriting test_library.gpr


In [69]:
# Re-build the library just in case
!gprbuild -P lattice.gpr

# Create object directory for the test project
!mkdir -p obj_test

# Build the test project.
!gprbuild -P test_library.gpr

# Run the executable
!./obj_test/main_library_test


gprbuild: "main_library_test" up to date
--- Testing Precompiled Library ---
Float Meet (Min) of 3.14 and 2.71:  2.71000E+00
Float Join (Max) of 3.14 and 2.71:  3.14000E+00
